# 📊 Data Exploration

Explore the violence detection datasets before training.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import json

from src.utils.config_parser import load_config
from src.preprocessing.utils import get_video_info, count_videos

%matplotlib inline

## 1. Load Configuration

In [ ]:
cfg = load_config("../configs/dataset_paths.yaml")
cfg_dict = cfg.to_dict()

print("Available datasets:")
for name in cfg_dict['datasets'].keys():
    print(f"  - {name}")

## 2. Dataset Statistics

In [ ]:
def analyze_dataset(dataset_name):
    """Analyze a dataset and print statistics."""
    dataset_cfg = cfg_dict['datasets'][dataset_name]
    raw_path = Path(dataset_cfg['raw_path'])
    
    print(f"\n{'='*50}")
    print(f"Dataset: {dataset_cfg['name']}")
    print(f"{'='*50}")
    
    if not raw_path.exists():
        print(f"❌ Not found: {raw_path}")
        return
    
    for class_info in dataset_cfg['classes']:
        class_path = raw_path / class_info['path']
        if not class_path.exists():
            continue
            
        videos = []
        for ext in dataset_cfg['video_extensions']:
            videos.extend(list(class_path.glob(f"*{ext}")))
        
        print(f"\n{class_info['name']}: {len(videos)} videos")
        
        # Sample video info
        if videos:
            sample = videos[0]
            info = get_video_info(str(sample))
            print(f"  Sample: {sample.name}")
            print(f"    Resolution: {info.get('width', '?')}x{info.get('height', '?')}")
            print(f"    FPS: {info.get('fps', '?'):.2f}")
            print(f"    Duration: {info.get('duration', '?'):.2f}s")
            print(f"    Frames: {info.get('frame_count', '?')}")

# Analyze all datasets
for name in cfg_dict['datasets'].keys():
    analyze_dataset(name)

## 3. Visualize Sample Frames

In [ ]:
def show_sample_frames(dataset_name, class_name, num_frames=4):
    """Display sample frames from a dataset class."""
    dataset_cfg = cfg_dict['datasets'][dataset_name]
    raw_path = Path(dataset_cfg['raw_path'])
    
    # Find class directory
    class_path = None
    for c in dataset_cfg['classes']:
        if c['name'] == class_name:
            class_path = raw_path / c['path']
            break
    
    if not class_path or not class_path.exists():
        print(f"❌ Class not found: {class_name}")
        return
    
    # Get videos
    videos = []
    for ext in dataset_cfg['video_extensions']:
        videos.extend(list(class_path.glob(f"*{ext}")))
    
    if not videos:
        print("❌ No videos found")
        return
    
    # Sample video
    video_path = videos[0]
    cap = cv2.VideoCapture(str(video_path))
    
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
    
    cap.release()
    
    # Plot
    fig, axes = plt.subplots(1, len(frames), figsize=(16, 4))
    if len(frames) == 1:
        axes = [axes]
    
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis('off')
    
    plt.suptitle(f"{dataset_name} - {class_name}\n{video_path.name}")
    plt.tight_layout()
    plt.show()

# Show samples from Violence in Car dataset
show_sample_frames('violence_in_car', 'Violence')
show_sample_frames('violence_in_car', 'NonViolence')

## 4. Check Processed Data

In [ ]:
def check_processed_data(dataset_name):
    """Check if preprocessed data exists."""
    dataset_cfg = cfg_dict['datasets'][dataset_name]
    processed_path = Path(dataset_cfg['processed_path'])
    
    print(f"\n{'='*50}")
    print(f"Processed Data: {dataset_name}")
    print(f"{'='*50}")
    
    if not processed_path.exists():
        print(f"❌ Not found: {processed_path}")
        print("Run preprocessing first!")
        return
    
    for split in ['train', 'val', 'test']:
        split_dir = processed_path / split / 'clips'
        if split_dir.exists():
            clips = list(split_dir.glob("*.npy"))
            print(f"{split}: {len(clips)} clips")
        else:
            print(f"{split}: Not found")

check_processed_data('violence_in_car')
check_processed_data('scvd')

## 5. Visualize Processed Clips

In [ ]:
def visualize_clip(clip_path):
    """Visualize a processed clip."""
    clip = np.load(clip_path)
    
    # Show first, middle, last frame
    indices = [0, len(clip)//2, -1]
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles = ['First', 'Middle', 'Last']
    
    for ax, idx, title in zip(axes, indices, titles):
        ax.imshow(clip[idx])
        ax.set_title(title)
        ax.axis('off')
    
    # Load metadata
    meta_path = Path(clip_path).with_suffix('.json')
    if meta_path.exists():
        with open(meta_path, 'r') as f:
            meta = json.load(f)
        label = "Violence" if meta['label'] == 1 else "Normal"
        plt.suptitle(f"Clip: {meta['clip_id']} | Label: {label}")
    
    plt.tight_layout()
    plt.show()

# Find a sample clip
processed_path = Path("../data/processed/violence_in_car/train/clips")
if processed_path.exists():
    clips = list(processed_path.glob("*.npy"))
    if clips:
        visualize_clip(clips[0])
    else:
        print("No clips found. Run preprocessing first!")
else:
    print("Processed data not found. Run preprocessing first!")

## 6. Class Distribution

In [ ]:
def plot_class_distribution(dataset_name):
    """Plot class distribution for a dataset."""
    dataset_cfg = cfg_dict['datasets'][dataset_name]
    raw_path = Path(dataset_cfg['raw_path'])
    
    if not raw_path.exists():
        print(f"❌ Dataset not found: {raw_path}")
        return
    
    class_counts = {}
    for class_info in dataset_cfg['classes']:
        class_path = raw_path / class_info['path']
        if not class_path.exists():
            continue
        
        videos = []
        for ext in dataset_cfg['video_extensions']:
            videos.extend(list(class_path.glob(f"*{ext}")))
        
        class_counts[class_info['name']] = len(videos)
    
    # Plot
    plt.figure(figsize=(8, 5))
    plt.bar(class_counts.keys(), class_counts.values(), color=['green', 'red', 'orange'])
    plt.title(f"{dataset_name} - Class Distribution")
    plt.xlabel("Class")
    plt.ylabel("Number of Videos")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print("\nClass counts:")
    for name, count in class_counts.items():
        print(f"  {name}: {count}")

plot_class_distribution('violence_in_car')
plot_class_distribution('scvd')

## Next Steps

1. ✅ Explore data (this notebook)
2. ⏳ Run preprocessing (`scripts/run_preprocessing.sh`)
3. ⏳ Train models (`scripts/run_training.sh`)
4. ⏳ Evaluate models (`scripts/run_evaluation.sh`)
5. ⏳ Deploy (`src/inference/real_time_detection.py`)